# SAM3 fine-tuning on SageMaker TrainingJobs

This notebook launches a SageMaker TrainingJob that fine-tunes SAM3 on
AWS_SAM using the recipe in `sam3/train/configs/aws_sam/aws_sam_finetune.yaml`.

**Inputs** (S3 channels passed to the job):

- `train`  → contains both `AWS_SAM/` (images + LabelMe JSONs) and
  `AWS_SAM_split/` (COCO train.json / test.json from `prepare_aws_sam.py`)
- `pretrained` → contains `sam3.pt` (the local SAM3 weights)

**Outputs** (two S3 locations, different semantics):

- `s3://<bucket>/<prefix>/checkpoints/aws_sam_finetune/checkpoints/`
  — **live per-epoch snapshots + tensorboard logs**, streamed
  continuously by SageMaker's `checkpoint_s3_uri` sync throughout the
  job. Includes `checkpoint.pt` (latest) and `checkpoint_<epoch>.pt`
  (per-epoch if `save_freq: 1` in the yaml). Survives job crashes; safe
  to inspect while the job is still running. Also what SageMaker uses
  to auto-resume on spot-instance preemption.
- `s3://<bucket>/<prefix>/output/<job_name>/output/model.tar.gz` —
  **one-time upload at job end**. Contains `checkpoint_merged.pt` (the
  raw fine-tuned weights merged with the pretrained SAM2 tracker into
  a self-contained file) + the BPE vocab. Ready to download and pass
  to eval / SageMaker deploy.

## Prerequisites

```bash
pip install sagemaker boto3
aws configure   # or assume an IAM role with sagemaker:* + s3 access
```

Run this notebook on a machine that has the SAM3 source tree (so the
SDK can package it) and AWS credentials. A SageMaker notebook
instance or your laptop both work.

In [1]:
# Pin sagemaker<3 — the v3 SDK (sagemaker>=3.0) is a major rewrite that
# moves PyTorch estimator + Session + get_execution_role out of their
# canonical paths. This notebook targets the long-stable v2 API.
%pip install --quiet 'sagemaker>=2.230,<3' boto3
# IMPORTANT: after installing, restart the kernel before running the
# next cell — otherwise the previously-imported v3 module stays cached.


Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ---------- Configuration: edit these ----------
import os
import sagemaker
import boto3

# Sanity-check SDK version. The v3 SDK has a different layout and this
# notebook is written against v2. The cell above pins sagemaker<3.
_v = getattr(sagemaker, "__version__", "unknown")
if _v.startswith("3.") or _v.startswith("4."):
    raise RuntimeError(
        f"Detected sagemaker {_v}, but this notebook requires v2.x. "
        f"Run the previous cell, then RESTART THE KERNEL, then retry."
    )
print("sagemaker version:", _v)

# v2-style imports (re-exported at top-level).
from sagemaker import get_execution_role

sess = sagemaker.Session()
role = get_execution_role()
sagemaker_default_bucket = sess.default_bucket()
region = sess.boto_session.region_name
print("sagemaker_default_bucket:", sagemaker_default_bucket)
print("sagemaker_region:", region)

S3_PREFIX        = "projects/sam3/data/aws_sam_finetune_v1"

# Local paths (these get uploaded to s3 once at launch time)
LOCAL_REPO_ROOT  = "/home/ec2-user/SageMaker/efs/Projects/sam3"
LOCAL_DATA       = "/home/ec2-user/SageMaker/efs/Projects/sam3/data"             # contains AWS_SAM/ and AWS_SAM_split/
LOCAL_PRETRAINED = "/home/ec2-user/SageMaker/efs/Models/sam3"                    # contains sam3.pt

# Instance & training settings
INSTANCE_TYPE    = "ml.p4de.24xlarge"     # 8 × A100 80GB. Use ml.p5.48xlarge for 8 × H100.
# Multi-node training: bump INSTANCE_COUNT to 2 (or more). SageMaker starts
# one container per instance sharing SM_HOSTS. Inside the container,
# train_entry.py reads SM_HOSTS + SM_CURRENT_HOST, translates them into
# SAM3_MASTER_ADDR / SAM3_NODE_RANK / SAM3_NUM_NODES, and passes
# --num-nodes to sam3/train/train.py. All ranks rendezvous over TCP.
#
# Caveats:
#   - DDP backend is `gloo` (see aws_sam_finetune_sm.yaml) — correct but
#     ignores p4de's 400 Gbps EFA. Switch to `nccl` + FI_* env vars once
#     you validate the cu128 NCCL wheel is stable.
#   - AWS_SAM has only ~4200 images; dataloader saturates before compute
#     so 2 nodes ≠ 2× throughput. Multi-node is mostly a scaling test here.
INSTANCE_COUNT   = 1                     # ← set to 2 for two-node training
NUM_GPUS         = 8                     # per-instance
MAX_RUNTIME_S    = 24 * 3600
VOLUME_SIZE_GB   = 500

# Hyperparameter overrides.
#
# IMPORTANT: editing sam3/train/configs/aws_sam/aws_sam_finetune_sm.yaml
# by hand DOES NOT WORK for SageMaker. That file is regenerated every
# job start by train_entry.py::write_runtime_config, which copies from
# the TEMPLATE `aws_sam_finetune.yaml` (hardcoded default in
# train_entry.py) and only rewrites the SageMaker-channel paths. To
# override a hyperparameter for SageMaker, put it in this HP dict —
# train_entry.py forwards these as `--train-batch-size 1`-style CLI
# args and patches the derived YAML in-place before invoking the
# trainer.
HP = {
    "num-gpus":           NUM_GPUS,
    # Starting small to isolate the OOM cause. p4de fit batch=8 locally
    # but OOM'd on SageMaker ml.p4de.24xlarge — bump this back up once
    # a batch=1 run confirms the pipeline is otherwise healthy.
    # "train-batch-size":   8,
    # Optional — set/uncomment to override other config defaults:
    # "max-epochs":         20,
    # "lr-scale":           0.1,
    # "num-train-workers":  0,
}

# Job name (timestamp will be appended automatically by the SDK)
JOB_NAME_PREFIX = "sam3-aws-sam-ft"

print("repo root:", LOCAL_REPO_ROOT)
print("HP overrides:", HP)
print(f"instances:  {INSTANCE_COUNT} × {INSTANCE_TYPE}  ({NUM_GPUS} GPUs each)")
print(f"total GPUs: {INSTANCE_COUNT * NUM_GPUS}")


## 1. Upload data + pretrained weights to S3

The training job needs the dataset and pretrained checkpoint pre-staged
in S3. This step is **one-time** — subsequent runs reuse the same S3 paths.

In [3]:
import boto3
import sagemaker
from sagemaker.s3 import S3Uploader

boto_session = boto3.Session(region_name=region)
sagemaker_session = sagemaker.Session(boto_session=boto_session)

s3_train_uri      = f"s3://{sagemaker_default_bucket}/{S3_PREFIX}/input/train"
s3_pretrained_uri = f"s3://{sagemaker_default_bucket}/{S3_PREFIX}/input/pretrained"
s3_output_uri     = f"s3://{sagemaker_default_bucket}/{S3_PREFIX}/output"

print("train data    →", s3_train_uri)
print("pretrained    →", s3_pretrained_uri)
print("output        →", s3_output_uri)

train data    → s3://sagemaker-us-west-2-452145973879/projects/sam3/data/aws_sam_finetune_v1/input/train
pretrained    → s3://sagemaker-us-west-2-452145973879/projects/sam3/data/aws_sam_finetune_v1/input/pretrained
output        → s3://sagemaker-us-west-2-452145973879/projects/sam3/data/aws_sam_finetune_v1/output


In [4]:
# # Upload the dataset (AWS_SAM/ + AWS_SAM_split/). Comment out after the
# # first successful upload — the data is large.
# S3Uploader.upload(
#     local_path=LOCAL_DATA,
#     desired_s3_uri=s3_train_uri,
#     sagemaker_session=sagemaker_session,
# )
# print("✓ data uploaded")

In [5]:
# # Upload the pretrained checkpoint (~9 GB). Comment out after the
# # first successful upload.
# S3Uploader.upload(
#     local_path=LOCAL_PRETRAINED,
#     desired_s3_uri=s3_pretrained_uri,
#     sagemaker_session=sagemaker_session,
# )
# print("✓ pretrained uploaded")

## 2. Launch the TrainingJob

SageMaker's PyTorch estimator packages the local source tree (the
sam3 repo) and ships it to the container, then runs `train_entry.py`
as the entry point. The script installs deps, patches the config to
use SageMaker channel paths, runs `sam3/train/train.py`, then merges
the resulting checkpoint and writes it to `/opt/ml/model/`.

In [6]:
# Build a slim source-bundle directory that the SageMaker PyTorch
# estimator will actually upload.
#
# WHY: SageMaker SDK v2 does NOT honor .sagemakerignore (that's v3-only).
# If we pass source_dir=LOCAL_REPO_ROOT, the SDK tarballs the ENTIRE
# repo including data/ (multi-GB) and runs/ (9GB checkpoints each),
# which silently hangs the upload for tens of minutes and then aborts
# with an opaque error (fit() returns None instead of a job handle).
#
# Instead, we assemble a small staging dir with just the code we need
# and point source_dir at that.
import os
import shutil

# Repo-local staging under tmp/ (which is in .gitignore, so it never
# pollutes git status and gets wiped by `git clean -xdf`).
STAGING_DIR = os.path.join(LOCAL_REPO_ROOT, "tmp", "s3_sagemaker_staging")

# Wipe any previous staging.
if os.path.exists(STAGING_DIR):
    shutil.rmtree(STAGING_DIR)
os.makedirs(STAGING_DIR)

# Include:
#   sam3/         — the actual package (source + assets/bpe*.txt.gz)
#   sagemaker/    — this launcher + train_entry.py + deploy scripts
#   scripts/      — merge_checkpoint.py + finetune helpers (train_entry
#                   calls scripts/finetune/merge_checkpoint.py after
#                   training completes)
#   pyproject.toml, MANIFEST.in, README.md, LICENSE — package metadata
INCLUDES = [
    "sam3",
    "sagemaker",
    "scripts",
    "pyproject.toml",
    "MANIFEST.in",
    "README.md",
    "LICENSE",
]
for name in INCLUDES:
    src = os.path.join(LOCAL_REPO_ROOT, name)
    dst = os.path.join(STAGING_DIR, name)
    if not os.path.exists(src):
        print(f"  skip (missing): {name}")
        continue
    if os.path.isdir(src):
        # Filtered copy — skip caches and other cruft that would bloat
        # the tarball but aren't needed at training time.
        shutil.copytree(
            src, dst,
            ignore=shutil.ignore_patterns(
                "__pycache__", "*.pyc", "*.pyo",
                ".ipynb_checkpoints", ".pytest_cache", ".mypy_cache",
                "*.egg-info", "build", "dist",
                # The launcher notebook itself doesn't need to ship —
                # it's the local orchestrator, not container code.
                "launch_training.ipynb",
            ),
        )
    else:
        shutil.copy2(src, dst)

# Verify staging size.
total = 0
for root, dirs, files in os.walk(STAGING_DIR):
    for f in files:
        total += os.path.getsize(os.path.join(root, f))
print(f"staging dir:  {STAGING_DIR}")
print(f"staging size: {total / 1e6:.1f} MB")

# From here forward, source_dir points at the staging dir, not the repo.
LOCAL_REPO_ROOT_FOR_SAGEMAKER = STAGING_DIR
print(f"source_dir for SageMaker: {LOCAL_REPO_ROOT_FOR_SAGEMAKER}")

staging dir:  /home/ec2-user/SageMaker/efs/Projects/sam3/tmp/s3_sagemaker_staging
staging size: 10.1 MB
source_dir for SageMaker: /home/ec2-user/SageMaker/efs/Projects/sam3/tmp/s3_sagemaker_staging


In [ ]:
from sagemaker.pytorch import PyTorch
from sagemaker.inputs import TrainingInput

# Per-epoch checkpoints stream to this S3 prefix continuously. SageMaker
# runs a background rsync from `checkpoint_local_path` (inside the
# container) to `checkpoint_s3_uri` throughout the job — so a crash
# mid-training doesn't lose everything, and you can browse per-epoch
# snapshots in S3 without waiting for job end.
#
# train_entry.py rewrites experiment_log_dir in the YAML to point at
# `/opt/ml/checkpoints/aws_sam_finetune`, so the trainer's
# `checkpoint.save_dir` = `/opt/ml/checkpoints/aws_sam_finetune/checkpoints`
# — that's the tree SageMaker mirrors.
s3_checkpoint_uri = f"s3://{sagemaker_default_bucket}/{S3_PREFIX}/checkpoints"

estimator = PyTorch(
    entry_point="train_entry.py",
    # Use the STAGING dir, not the full repo — SDK v2 doesn't honor
    # .sagemakerignore, so pointing at the repo root uploads data/,
    # runs/, .git/, etc. and hangs. The previous cell built a slim
    # copy at tmp/s3_sagemaker_staging/ with just what we need.
    source_dir=LOCAL_REPO_ROOT_FOR_SAGEMAKER,
    role=role,
    framework_version="2.4.0",                        # cu124 base image
    py_version="py311",
    instance_type=INSTANCE_TYPE,
    instance_count=INSTANCE_COUNT,
    volume_size=VOLUME_SIZE_GB,
    max_run=MAX_RUNTIME_S,
    output_path=s3_output_uri,
    # Continuous per-epoch checkpoint sync to S3. Anything the trainer
    # writes under checkpoint_local_path is streamed to checkpoint_s3_uri
    # in the background — surviving crashes AND resumable across spot
    # interruptions if you later flip use_spot_instances=True.
    checkpoint_s3_uri=s3_checkpoint_uri,
    checkpoint_local_path="/opt/ml/checkpoints",
    base_job_name=JOB_NAME_PREFIX,
    sagemaker_session=sagemaker_session,
    hyperparameters=HP,                                # forwarded as CLI args
    environment={
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        # SageMaker chdirs into /opt/ml/code/ inside the container;
        # that's where our staging tree lands.
        "SAM3_REPO_ROOT": "/opt/ml/code",
    },
    disable_profiler=True,
    # Tip: set this if you want fast restart on interruption.
    keep_alive_period_in_seconds=1800,
)

# The SDK looks for source_dir/entry_point. Because train_entry.py
# lives under sagemaker/train/ inside the staging dir, tell the SDK
# to look there:
estimator.entry_point = "sagemaker/train/train_entry.py"

print("Estimator built. ImageURI:", estimator.training_image_uri())
print("per-epoch checkpoints → ", s3_checkpoint_uri)
print("final model.tar.gz    → ", s3_output_uri, "/<job>/output/model.tar.gz")


In [8]:
# Kick it off. estimator.fit() is blocking — pass wait=False if you
# want the cell to return immediately and monitor via the SageMaker
# console.
estimator.fit(
    inputs={
        "train":      TrainingInput(s3_data=s3_train_uri),
        "pretrained": TrainingInput(s3_data=s3_pretrained_uri),
    },
    wait=True,        # set False to background the job
    logs="All",       # stream all container logs to this notebook
)
print("Job name:", estimator.latest_training_job.job_name)
print("Model artifact:", estimator.model_data)

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: sam3-aws-sam-ft-2026-07-01-12-39-59-799


2026-07-01 12:40:01 Starting - Starting the training job...
2026-07-01 12:40:28 Pending - Training job waiting for capacity...
2026-07-01 12:40:53 Pending - Preparing the instances for training........................
2026-07-01 12:44:52 Downloading - Downloading input data.....Job name: sam3-aws-sam-ft-2026-07-01-12-39-59-799


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:13                                                                                   │
│                                                                                                  │
│   10 │   logs="All",       # stream all container logs to this notebook                          │
│   11 )                                                                                           │
│   12 print("Job name:", estimator.latest_training_job.job_name)                                  │
│ ❱ 13 print("Model artifact:", estimator.model_data)                                              │
│   14                                                                                             │
│                                                                                                  │
│ /home/ec2-user/SageMaker/efs/conda_envs/sam3/lib/python3.12/site-packages/sagemaker/estimator.py │
│ :1948 in model_data                                                                              │
│                                                                                                  │
│   1945 │   │   │   job_details = self.sagemaker_session.sagemaker_client.describe_training_job(  │
│   1946 │   │   │   │   TrainingJobName=self.latest_training_job.name                             │
│   1947 │   │   │   )                                                                             │
│ ❱ 1948 │   │   │   model_uri = job_details["ModelArtifacts"]["S3ModelArtifacts"]                 │
│   1949 │   │   │   compression_type = job_details.get("OutputDataConfig", {}).get(               │
│   1950 │   │   │   │   "CompressionType", "GZIP"                                                 │
│   1951 │   │   │   )                                                                             │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
KeyError: 'ModelArtifacts'

## 3. Download the merged checkpoint back to local disk

SageMaker uploads anything written to `/opt/ml/model/` as a
`model.tar.gz` in the output S3 path. `train_entry.py` writes
`checkpoint_merged.pt` (the self-contained fine-tuned weights) +
the BPE vocab there.

In [ ]:
# from sagemaker.s3 import S3Downloader

# # estimator.model_data is the s3 URI of model.tar.gz
# local_artifact = "runs/sagemaker_artifact"
# os.makedirs(local_artifact, exist_ok=True)
# S3Downloader.download(
#     s3_uri=estimator.model_data,
#     local_path=local_artifact,
#     sagemaker_session=sagemaker_session,
# )
# print("downloaded to:", local_artifact)

# # Unpack
# import tarfile
# tar_path = os.path.join(local_artifact, "model.tar.gz")
# with tarfile.open(tar_path) as t:
#     t.extractall(local_artifact)
# print("contents:", os.listdir(local_artifact))

## 4. Run eval against the downloaded checkpoint

The merged file is identical to what `merge_checkpoint.py` would
produce locally, so you can plug it straight into the eval harness:

```bash
python scripts/finetune/eval/evaluate_interactive.py \
    --coco data/AWS_SAM_split/test.json \
    --image-root data/AWS_SAM \
    --checkpoint runs/sagemaker_artifact/checkpoint_merged.pt \
    --output runs/eval/finetuned_sagemaker.json
```

## Notes / gotchas

- **First run is slow** because `pip install -e ".[dev,train]"` runs
  inside the container. If you do many runs, consider building a
  custom Docker image that has SAM3 pre-installed and pass it via
  `image_uri=...` on the estimator.
- **S3 charges and bandwidth**: data + pretrained ≈ 10 GB, model
  artifact ≈ 9 GB. Cleanup with `aws s3 rm --recursive` when done.
- **Spot training**: add `use_spot_instances=True, max_wait=...` to
  the estimator to drop costs ~60% on long jobs at the risk of
  preemption. With `checkpoint_s3_uri` wired (this cell), SageMaker
  auto-resumes from the last per-epoch checkpoint on preemption.
- **Multi-instance**: set `INSTANCE_COUNT > 1` on the config cell —
  the entry point + `sam3/train/train.py` are wired for cross-node
  DDP via `SM_HOSTS`. Backend is still `gloo`; see the caveats note
  next to `INSTANCE_COUNT` for the NCCL/EFA follow-up.